# Étape 2 – Prévisions météo à 7 jours (OpenWeatherMap One Call API 4.0)

**Objectif** : récupérer, pour chacune des 35 destinations, les prévisions météo des prochains jours,
puis définir un **score « beau temps »** pour classer les destinations.

**Source** : [OpenWeatherMap One Call API 4.0](https://openweathermap.org/api/one-call-4), endpoint
`/data/4.0/onecall/timeline/1day` (prévisions jour par jour).

> **Écart avec l'énoncé** : l'énoncé Jedha cite One Call API **3.0**. Depuis juin 2026, OpenWeather
> propose la version **4.0** aux nouveaux abonnés. Le code suit donc la documentation actuelle :
> les principes (appel REST, JSON, DataFrame) sont identiques, seuls l'URL et la structure de la réponse changent.

**Compétence du référentiel** : C3 – collecter des données provenant de différentes sources (API web).

**Entrée** : `data/raw/cities.csv` (étape 1) – **Sortie** : `data/raw/weather_daily.csv`

**Démarche en deux temps** :
1. **Un seul appel de test** (Paris) pour lire la réponse brute et identifier les champs réellement disponibles ;
2. Ensuite seulement : la boucle sur les 35 villes et la définition du score.

## 1. Imports, chemins et clé API

In [1]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv


def find_project_root(start: Path) -> Path:
    """Remonte l'arborescence jusqu'au dossier qui contient requirements.txt (racine du projet)."""
    for folder in [start, *start.parents]:
        if (folder / "requirements.txt").exists():
            return folder
    raise FileNotFoundError("Racine du projet introuvable (requirements.txt absent)")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CITIES_FILE = RAW_DIR / "cities.csv"

# La clé est lue depuis le fichier .env (exclu de Git) : elle n'apparaît jamais dans le code
load_dotenv(PROJECT_ROOT / ".env")
OWM_API_KEY = os.getenv("OWM_API_KEY")
assert OWM_API_KEY, "OWM_API_KEY absente du fichier .env"
print("Clé API chargée :", "oui" if OWM_API_KEY else "non")  # on n'affiche jamais la clé elle-même

Clé API chargée : oui


## 2. Chargement des destinations (sortie de l'étape 1)

In [2]:
cities = pd.read_csv(CITIES_FILE)
print(f"{len(cities)} destinations chargées")
cities[["city_id", "city_name", "latitude", "longitude"]].head()

35 destinations chargées


,city_id,city_name,latitude,longitude
0,1,Mont Saint Michel,48.635954,-1.511460
1,2,St Malo,48.649518,-2.026041
2,3,Bayeux,49.276462,-0.702474
3,4,Le Havre,49.493898,0.107973
4,5,Rouen,49.440459,1.093966


## 3. Fonction d'appel à l'API

Paramètres :
- `units=metric` : températures en °C (par défaut, l'API renvoie des kelvins) ;
- `lang=fr` : descriptions météo en français ;
- `appid` : la clé API, transmise en paramètre.

⚠️ **Sécurité** : la clé faisant partie de l'URL appelée, on n'affiche **jamais** `response.url` ni les liens
renvoyés par l'API sans les masquer (fonction `mask_key`).

In [3]:
OWM_DAILY_URL = "https://api.openweathermap.org/data/4.0/onecall/timeline/1day"


def mask_key(text: str) -> str:
    """Remplace la clé API par des étoiles dans un texte (URL, message d'erreur...)."""
    return str(text).replace(OWM_API_KEY, "***")


def fetch_daily_forecast(lat: float, lon: float) -> requests.Response:
    """Appelle l'endpoint 'timeline/1day' de One Call API 4.0 pour un point GPS."""
    params = {"lat": lat, "lon": lon, "appid": OWM_API_KEY, "units": "metric", "lang": "fr"}
    return requests.get(OWM_DAILY_URL, params=params, timeout=15)

## 4. Appel de test sur Paris et lecture de la réponse brute

On regarde successivement :
1. le **code HTTP** (200 = succès ; 401 = clé refusée ou pas encore activée ; 429 = trop d'appels) ;
2. la **structure générale** de la réponse (clés de premier niveau) ;
3. **un enregistrement journalier complet**, pour savoir quels champs utiliser (température, `pop`, `rain`…) ;
4. les **dates couvertes**, pour vérifier que la prévision commence bien aujourd'hui.

In [4]:
paris = cities.loc[cities["city_name"] == "Paris"].iloc[0]
response = fetch_daily_forecast(paris["latitude"], paris["longitude"])

print("Code HTTP :", response.status_code)
if response.status_code != 200:
    print("Message :", mask_key(response.text[:500]))

Code HTTP : 200


In [5]:
# Structure générale de la réponse : clés de premier niveau et type de chaque valeur
payload = response.json()
print("Type de la réponse :", type(payload).__name__)
if isinstance(payload, dict):
    for key, value in payload.items():
        size = f" ({len(value)} éléments)" if isinstance(value, (list, dict)) else ""
        print(f"- {key}: {type(value).__name__}{size}")

Type de la réponse : dict
- lat: float
- lon: float
- timezone: str
- timezone_offset: int
- data: list (10 éléments)
- prev: str
- next: str


In [6]:
# Repérage automatique de la liste des enregistrements journaliers (la première valeur de type liste)
records = payload if isinstance(payload, list) else next(
    (value for value in payload.values() if isinstance(value, list)), []
)
print(f"{len(records)} enregistrements journaliers")

# Affichage complet du premier enregistrement (masqué au cas où la clé apparaîtrait)
print(mask_key(json.dumps(records[0], indent=2, ensure_ascii=False)) if records else "Aucun enregistrement")

10 enregistrements journaliers
{
  "dt": 1793750400,
  "sunrise": 1793774519,
  "sunset": 1793809592,
  "moonrise": 1793754480,
  "moonset": 1793802000,
  "moon_phase": 0.84,
  "temp": {
    "day": 9.51,
    "min": 5.14,
    "max": 10.64,
    "night": 5.94,
    "eve": 9.49,
    "morn": 5.14
  },
  "feels_like": {
    "day": 9.51,
    "night": 5.94,
    "eve": 9.49,
    "morn": 5.14
  },
  "pressure": 1023.98,
  "humidity": 56,
  "dew_point": 1.32,
  "wind_speed": 5.56,
  "wind_deg": 72,
  "weather": [
    {
      "id": 803,
      "main": "Clouds",
      "description": "nuageux",
      "icon": "04d"
    }
  ],
  "clouds": 73,
  "pop": 0,
  "uvi": 1.42
}


In [7]:
# Dates couvertes par la réponse (dt = horodatage Unix en secondes, UTC)
for record in records:
    day = datetime.fromtimestamp(record["dt"], tz=timezone.utc).strftime("%a %d/%m/%Y")
    rain = record.get("rain", "absent")
    print(f"{day} | temp.day = {record.get('temp', {}).get('day')} | pop = {record.get('pop')} | rain = {rain}")

Wed 04/11/2026 | temp.day = 9.51 | pop = 0 | rain = absent
Thu 05/11/2026 | temp.day = 8.93 | pop = 0.03 | rain = absent
Fri 06/11/2026 | temp.day = 10.49 | pop = 0.23 | rain = 0.36
Sat 07/11/2026 | temp.day = 11.68 | pop = 0 | rain = absent
Sun 08/11/2026 | temp.day = 11.15 | pop = 0.03 | rain = absent
Mon 09/11/2026 | temp.day = 10.18 | pop = 0.46 | rain = 0.36
Tue 10/11/2026 | temp.day = 8.7 | pop = 0.91 | rain = 18.83
Wed 11/11/2026 | temp.day = 8.7 | pop = 0.95 | rain = 7.9
Thu 12/11/2026 | temp.day = 8.25 | pop = 0 | rain = absent
Fri 13/11/2026 | temp.day = 7.61 | pop = 0 | rain = absent


## 5. Analyse de la réponse et choix du score « beau temps »

**Constats sur la réponse de test (Paris)** :
- `data` contient **10 jours** à partir d'aujourd'hui : **un seul appel par ville** suffit (35 appels au total) ;
- `rain` est le **cumul de pluie de la journée en mm** ; le champ est **absent les jours sans pluie**
  → il sera remplacé par **0** (nettoyage) ;
- `pop` est la probabilité de pluie (0 à 1) ; `temp.day` la température en journée (°C, grâce à `units=metric`) ;
- `timezone_offset` (en secondes) permet de calculer les dates en **heure locale**.

**Choix retenus** :
- **Fenêtre** : les 7 jours **à partir de demain (J+1 à J+7)**, la journée en cours étant déjà entamée ;
- **Indicateurs par ville** (sur ces 7 jours) : `total_rain_mm` (somme), `avg_pop` (moyenne), `avg_temp_day` (moyenne) ;
- **Score** (de 0 à 1, plus il est élevé, plus il fera beau), après normalisation min-max sur les 35 villes :

`weather_score = 0.5 × (1 − total_rain normalisé) + 0.2 × (1 − avg_pop) + 0.3 × avg_temp normalisée`

  - la pluie pèse 70 % (quantité + risque) : premier facteur qui gâche un séjour ;
  - la température pèse 30 % : en novembre, il vaut mieux qu'il fasse plus chaud
    (limite : pondération à revoir en été, où une chaleur excessive devient un défaut) ;
  - la normalisation rend comparables des grandeurs d'unités différentes (mm, probabilité, °C).

## 6. Extraction des champs utiles d'une réponse

La fonction transforme la réponse JSON d'une ville en **une ligne par jour**.
On conserve plus de champs que ceux du score (humidité, nuages, vent, description…) :
la donnée brute doit rester exploitable pour d'autres analyses futures.

In [8]:
def parse_daily_records(payload: dict, city_id: int, collected_at: str) -> list[dict]:
    """Convertit la réponse 'timeline/1day' d'une ville en une liste de lignes (une par jour)."""
    offset = payload.get("timezone_offset", 0)  # décalage horaire local, en secondes
    rows = []
    for record in payload.get("data", []):
        weather = (record.get("weather") or [{}])[0]  # la description est dans une liste d'un élément
        rows.append({
            "city_id": city_id,
            "forecast_date": datetime.fromtimestamp(record["dt"] + offset, tz=timezone.utc).date(),
            "temp_day": record.get("temp", {}).get("day"),
            "temp_min": record.get("temp", {}).get("min"),
            "temp_max": record.get("temp", {}).get("max"),
            "feels_like_day": record.get("feels_like", {}).get("day"),
            "humidity": record.get("humidity"),
            "pressure": record.get("pressure"),
            "clouds": record.get("clouds"),
            "wind_speed": record.get("wind_speed"),
            "pop": record.get("pop"),
            "rain_mm": record.get("rain", 0.0),  # champ absent = pas de pluie → 0 mm
            "uvi": record.get("uvi"),
            "weather_main": weather.get("main"),
            "weather_description": weather.get("description"),
            "collected_at": collected_at,  # date de collecte : une prévision dépend du jour où on la demande
        })
    return rows


# Vérification sur la réponse de test de Paris (déjà en mémoire)
pd.DataFrame(parse_daily_records(payload, int(paris["city_id"]), "test")).head(3)

,city_id,forecast_date,temp_day,temp_min,temp_max,feels_like_day,humidity,pressure,clouds,wind_speed,pop,rain_mm,uvi,weather_main,weather_description,collected_at
0,6,2026-11-04,9.51,5.14,10.64,9.51,56,1023.98,73,5.56,0.00,0.00,1.42,Clouds,nuageux,test
1,6,2026-11-05,8.93,4.60,12.97,8.93,61,1023.86,64,3.30,0.03,0.00,1.39,Clouds,nuageux,test
2,6,2026-11-06,10.49,9.45,11.23,10.49,65,1024.26,100,1.40,0.23,0.36,0.74,Rain,légère pluie,test


## 7. Collecte pour les 35 destinations

- 35 appels (un par ville), espacés d'une seconde : bien en dessous de la limite de 60 appels/minute ;
- une ville en échec n'interrompt pas la boucle : elle est listée à la fin pour être relancée si besoin.

In [9]:
import time

collected_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
all_rows, failures = [], []

for city in cities.itertuples():
    response = fetch_daily_forecast(city.latitude, city.longitude)
    if response.status_code == 200:
        rows = parse_daily_records(response.json(), city.city_id, collected_at)
        all_rows.extend(rows)
        print(f"{city.city_id:>2}. {city.city_name:<30} -> {len(rows)} jours")
    else:
        failures.append(city.city_name)
        print(f"{city.city_id:>2}. {city.city_name:<30} -> ⚠️ HTTP {response.status_code} : {mask_key(response.text[:150])}")
    time.sleep(1)

weather_daily = pd.DataFrame(all_rows)
print(f"\n{len(weather_daily)} lignes collectées – villes en échec : {failures or 'aucune'}")

 1. Mont Saint Michel              -> 10 jours
 2. St Malo                        -> 10 jours
 3. Bayeux                         -> 10 jours
 4. Le Havre                       -> 10 jours
 5. Rouen                          -> 10 jours
 6. Paris                          -> 10 jours
 7. Amiens                         -> 10 jours
 8. Lille                          -> 10 jours
 9. Strasbourg                     -> 10 jours
10. Chateau du Haut Koenigsbourg   -> 10 jours
11. Colmar                         -> 10 jours
12. Eguisheim                      -> 10 jours
13. Besancon                       -> 10 jours
14. Dijon                          -> 10 jours
15. Annecy                         -> 10 jours
16. Grenoble                       -> 10 jours
17. Lyon                           -> 10 jours
18. Gorges du Verdon               -> 10 jours
19. Bormes les Mimosas             -> 10 jours
20. Cassis                         -> 10 jours
21. Marseille                      -> 10 jours
22. Aix en Pr

### Contrôle qualité des données brutes
- 35 villes × 10 jours = **350 lignes** attendues ;
- pas de valeur manquante sur les champs du score ;
- valeurs plausibles (températures, probabilités entre 0 et 1, pluie positive).

In [10]:
print("Villes présentes :", weather_daily["city_id"].nunique(), "/ 35")
print("Jours par ville  :", weather_daily.groupby("city_id").size().unique())
print("\nValeurs manquantes (champs du score) :")
print(weather_daily[["temp_day", "pop", "rain_mm"]].isna().sum())
weather_daily[["temp_day", "temp_min", "temp_max", "humidity", "clouds", "pop", "rain_mm"]].describe().round(2)

Villes présentes : 35 / 35
Jours par ville  : [10]

Valeurs manquantes (champs du score) :
temp_day    0
pop         0
rain_mm     0
dtype: int64


,temp_day,temp_min,temp_max,humidity,clouds,pop,rain_mm
count,350.00,350.00,350.00,350.00,350.00,350.00,350.00
mean,12.49,8.26,13.62,70.94,67.66,0.40,4.17
std,3.33,3.99,3.30,12.85,37.17,0.40,10.49
min,2.65,-2.20,3.31,36.00,0.00,0.00,0.00
25%,9.93,5.17,10.99,62.00,35.50,0.00,0.00
50%,12.24,7.94,13.34,69.00,88.00,0.28,0.00
75%,15.36,11.47,16.16,80.00,99.00,0.86,3.60
max,20.44,16.27,22.61,98.00,100.00,1.00,90.68


In [11]:
WEATHER_DAILY_FILE = RAW_DIR / "weather_daily.csv"
weather_daily.to_csv(WEATHER_DAILY_FILE, index=False, encoding="utf-8")
print(f"Données brutes enregistrées : {WEATHER_DAILY_FILE} ({len(weather_daily)} lignes)")

Données brutes enregistrées : c:\Users\frede\OneDrive\Documents\_Jedha\2- dsfs-od-15 - Fullstack\Final Projects - dsfs-od-15\B1 - Kayak\kayak-plan-your-trip\data\raw\weather_daily.csv (350 lignes)


## 8. Indicateurs sur 7 jours et score « beau temps »

Pour chaque ville, les jours sont numérotés dans l'ordre chronologique (0 = aujourd'hui) ;
on ne garde que les jours **1 à 7** (J+1 à J+7).

In [12]:
WINDOW_START, WINDOW_END = 1, 7  # J+1 à J+7

window = weather_daily.sort_values(["city_id", "forecast_date"]).copy()
window["day_index"] = window.groupby("city_id").cumcount()  # 0 = aujourd'hui, 1 = demain...
window = window[window["day_index"].between(WINDOW_START, WINDOW_END)]

assert (window.groupby("city_id").size() == 7).all(), "Chaque ville doit avoir 7 jours dans la fenêtre"
print("Période analysée :", window["forecast_date"].min(), "→", window["forecast_date"].max())

summary = window.groupby("city_id").agg(
    period_start=("forecast_date", "min"),
    period_end=("forecast_date", "max"),
    total_rain_mm=("rain_mm", "sum"),
    avg_pop=("pop", "mean"),
    avg_temp_day=("temp_day", "mean"),
    rainy_days=("rain_mm", lambda s: int((s > 0).sum())),  # information complémentaire (hors score)
).reset_index()
summary.head()

Période analysée : 2026-11-05 → 2026-11-11


,city_id,period_start,period_end,total_rain_mm,avg_pop,avg_temp_day,rainy_days
0,1,2026-11-05,2026-11-11,31.83,0.291429,10.472857,2
1,2,2026-11-05,2026-11-11,9.00,0.287143,10.478571,2
2,3,2026-11-05,2026-11-11,1.51,0.245714,10.257143,3
3,4,2026-11-05,2026-11-11,3.44,0.188571,10.188571,1
4,5,2026-11-05,2026-11-11,0.00,0.117143,10.635714,0


In [13]:
def min_max(series: pd.Series) -> pd.Series:
    """Normalisation min-max : ramène les valeurs entre 0 (minimum) et 1 (maximum)."""
    value_range = series.max() - series.min()
    return (series - series.min()) / value_range if value_range else series * 0.0


WEIGHT_RAIN, WEIGHT_POP, WEIGHT_TEMP = 0.5, 0.2, 0.3

summary["weather_score"] = (
    WEIGHT_RAIN * (1 - min_max(summary["total_rain_mm"]))
    + WEIGHT_POP * (1 - summary["avg_pop"])          # déjà entre 0 et 1
    + WEIGHT_TEMP * min_max(summary["avg_temp_day"])
).round(3)
summary["weather_rank"] = summary["weather_score"].rank(ascending=False, method="min").astype(int)

# Ajout du nom et des coordonnées des villes (jointure sur la clé city_id)
weather_summary = (
    cities[["city_id", "city_name", "latitude", "longitude"]]
    .merge(summary, on="city_id", how="inner", validate="one_to_one")
    .sort_values("weather_rank")
    .round({"total_rain_mm": 1, "avg_pop": 2, "avg_temp_day": 1})
)
weather_summary.head(10)

,city_id,city_name,latitude,longitude,period_start,period_end,total_rain_mm,avg_pop,avg_temp_day,rainy_days,weather_score,weather_rank
27,28,Collioure,42.525050,3.083155,2026-11-05,2026-11-11,1.0,0.27,16.8,3,0.933,1
33,34,Bayonne,43.494514,-1.473666,2026-11-05,2026-11-11,5.6,0.26,17.0,3,0.914,2
28,29,Carcassonne,43.213036,2.349107,2026-11-05,2026-11-11,3.0,0.23,15.6,2,0.894,3
32,33,Biarritz,43.471144,-1.552728,2026-11-05,2026-11-11,5.6,0.30,16.5,4,0.890,4
30,31,Toulouse,43.604464,1.444243,2026-11-05,2026-11-11,3.7,0.24,15.4,2,0.882,5
31,32,Montauban,44.017584,1.354999,2026-11-05,2026-11-11,1.7,0.28,15.0,2,0.872,6
29,30,Ariege,42.945537,1.406554,2026-11-05,2026-11-11,1.5,0.20,12.3,3,0.811,7
4,5,Rouen,49.440459,1.093966,2026-11-05,2026-11-11,0.0,0.12,10.6,0,0.786,8
5,6,Paris,48.858890,2.320041,2026-11-05,2026-11-11,0.5,0.07,10.2,2,0.779,9
7,8,Lille,50.636565,3.063528,2026-11-05,2026-11-11,0.5,0.06,9.3,1,0.754,10


## 9. Top 5 des destinations et carte de contrôle

In [14]:
top5 = weather_summary.nsmallest(5, "weather_rank")
display(top5[["weather_rank", "city_name", "weather_score", "total_rain_mm", "avg_pop", "avg_temp_day", "rainy_days"]])

import plotly.express as px

fig = px.scatter_map(
    weather_summary,
    lat="latitude",
    lon="longitude",
    color="weather_score",
    size="weather_score",
    size_max=18,
    color_continuous_scale="RdYlGn",  # rouge = mauvais temps, vert = beau temps
    hover_name="city_name",
    hover_data={"weather_rank": True, "total_rain_mm": True, "avg_temp_day": True,
                "latitude": False, "longitude": False},
    zoom=4.3,
    height=650,
    map_style="open-street-map",
    title=f"Score « beau temps » du {weather_summary['period_start'].min()} au {weather_summary['period_end'].max()}",
)
fig.show()

,weather_rank,city_name,weather_score,total_rain_mm,avg_pop,avg_temp_day,rainy_days
27,1,Collioure,0.933,1.0,0.27,16.8,3
33,2,Bayonne,0.914,5.6,0.26,17.0,3
28,3,Carcassonne,0.894,3.0,0.23,15.6,2
32,4,Biarritz,0.890,5.6,0.30,16.5,4
30,5,Toulouse,0.882,3.7,0.24,15.4,2


### Analyse des résultats
- Collecte complète : 35 villes × 10 jours, aucune valeur manquante, valeurs plausibles pour la saison.
- Top 5 (du 05/11 au 11/11/2026) : Collioure, Bayonne, Carcassonne, Biarritz, Toulouse – Sud-Ouest et Roussillon épargnés par la pluie et plus doux.
- Bas du classement : Alpes (froid + pluie) et vallée du Rhône (épisode méditerranéen, jusqu'à 90 mm en une journée).
- **Limite du score** : la normalisation min-max est sensible aux valeurs extrêmes. Les forts cumuls de la vallée du Rhône écrasent l'échelle de la pluie, si bien que la température départage le haut du classement (ex. : Bayonne devant Carcassonne malgré plus de pluie).
  Améliorations possibles : normalisation par le rang, ou logarithme des cumuls de pluie.
- Le classement dépend de la date de collecte (colonne `collected_at`) : il doit être recalculé régulièrement pour rester pertinent.

## 10. Enregistrement des données transformées

`weather_summary.csv` est une donnée **transformée** (agrégats + score) : elle est rangée dans `data/processed/`,
et ira dans la zone `processed/` du Data Lake à l'étape 4.

In [15]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
WEATHER_SUMMARY_FILE = PROCESSED_DIR / "weather_summary.csv"

weather_summary.to_csv(WEATHER_SUMMARY_FILE, index=False, encoding="utf-8")
print(f"Données transformées enregistrées : {WEATHER_SUMMARY_FILE} ({len(weather_summary)} lignes)")

Données transformées enregistrées : c:\Users\frede\OneDrive\Documents\_Jedha\2- dsfs-od-15 - Fullstack\Final Projects - dsfs-od-15\B1 - Kayak\kayak-plan-your-trip\data\processed\weather_summary.csv (35 lignes)
